In [32]:
import random, pdb
from tqdm import tqdm

import pandas as pd
import numpy as np

from multiprocessing import Pool, cpu_count

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from scipy.stats import gaussian_kde

In [33]:
def compute_class_overlap(probs, labels, bandwidth='scott', num_points=1000):
    """
    Compute the overlap between the KDE-estimated probability distributions
    for the positive and negative classes.
    
    Parameters:
    - probs: array-like, predicted probabilities for the positive class (length = n_samples)
    - labels: array-like, true binary labels (0 or 1)
    - bandwidth: str or float, bandwidth method or value for KDE (default 'scott')
    - num_points: int, number of points for numerical integration (default 1000)

    Returns:
    - overlap: float, area under min(P_pos(x), P_neg(x)), ranges from 0 (no overlap) to 1 (complete)
    """
    probs = np.asarray(probs)
    labels = np.asarray(labels)

    probs_pos = probs[labels == 'P']
    probs_neg = probs[labels == 'N']

    if len(probs_pos) < 2 or len(probs_neg) < 2:
        raise ValueError("Need at least two samples per class for KDE.")

    kde_pos = gaussian_kde(probs_pos, bw_method=bandwidth)
    kde_neg = gaussian_kde(probs_neg, bw_method=bandwidth)

    x = np.linspace(0, 1, num_points)
    kde_vals_pos = kde_pos(x)
    kde_vals_neg = kde_neg(x)

    overlap = np.trapezoid(np.minimum(kde_vals_pos, kde_vals_neg), x)
    
    return overlap

# Function to generate predictions
def generate_prediction(df):
    # Split the data into features and target
    X = df.drop('class', axis=1)
    y = df['class']

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=40)
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Create and train the Logistic Regression model
    model = LogisticRegression(random_state=40, max_iter=1000)
    model.fit(X_train, y_train)

    # Evaluate the model using AUC metric
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    # Compute the class overlap
    overlap = compute_class_overlap(model.predict_proba(X_test)[:, 1], y_test)

    return auc, overlap

In [23]:
def process_row(row):
    pos_class = eval(row['Positive'])
    neg_class = eval(row['Negative'])
    easy_class = eval(row['Easy'])
    hard_class = eval(row['Hard'])

    # Create DataFrames for each scenario
    df_pos_neg = df[df['class'].isin(pos_class + neg_class)].copy()
    df_pos_neg.loc[:, 'class'] = df_pos_neg['class'].apply(lambda x: 'P' if x in pos_class else 'N')
    auc, overlap = generate_prediction(df_pos_neg)

    df_pos_easy = df[df['class'].isin(pos_class + easy_class)].copy()
    df_pos_easy.loc[:, 'class'] = df_pos_easy['class'].apply(lambda x: 'P' if x in pos_class else 'N')
    auc_easy, overlap_easy = generate_prediction(df_pos_easy)

    df_pos_hard = df[df['class'].isin(pos_class + hard_class)].copy()
    df_pos_hard.loc[:, 'class'] = df_pos_hard['class'].apply(lambda x: 'P' if x in pos_class else 'N')
    auc_hard, overlap_hard = generate_prediction(df_pos_hard)

            # Return the results as a dictionary
    salve = {
        'Positive': pos_class,
        'Negative': neg_class,
        'Easy': easy_class,
        'Hard': hard_class,
        'AUC': auc,
        'AUC_Easy': auc_easy,
        'AUC_Hard': auc_hard,
        'Overlap': overlap,
        'Overlap_Easy': overlap_easy,
        'Overlap_Hard': overlap_hard
    }
    print(salve)

    # Return the results as a dictionary
    return {
        'Positive': pos_class,
        'Negative': neg_class,
        'Easy': easy_class,
        'Hard': hard_class,
        'AUC': auc,
        'AUC_Easy': auc_easy,
        'AUC_Hard': auc_hard,
        'Overlap': overlap,
        'Overlap_Easy': overlap_easy,
        'Overlap_Hard': overlap_hard
    }

In [30]:
def running_esperiment_parallel(search_df):
    rows = [row for _, row in search_df.iterrows()]

    results = []

    for row in tqdm(rows, desc='Progress'):
        result = process_row(row)
        print(result)
        if result is not None:
            results.append(result)

    # Filter out None results (in case of errors)
    results = [res for res in results if res is not None]

    # Convert the results to a DataFrame
    results_df = pd.DataFrame(results)
    return results_df

In [34]:
# Load the search results
search_results_path = "./search/search_results_Covertype.csv"
search_results = pd.read_csv(search_results_path)
df = pd.read_csv("./datasets/Covertype.csv")

# Run the experiment in parallel
results_df = running_esperiment_parallel(search_results)

# Save the results to a new CSV file
results_df.to_csv("processed_results_Covertype.csv", index=False)
print("Processing complete. Results saved to 'processed_results_Covertype.csv'.")

Progress:   0%|          | 1/125000 [00:01<59:54:54,  1.73s/it]

{'Positive': ['Douglas_fir'], 'Negative': ['Spruce_Fir', 'Cottonwood_Willow', 'Krummholz', 'Lodgepole_Pine', 'Ponderosa_Pine'], 'Easy': ['Krummholz', 'Spruce_Fir'], 'Hard': ['Lodgepole_Pine', 'Cottonwood_Willow', 'Ponderosa_Pine'], 'AUC': np.float64(0.8572072445725651), 'AUC_Easy': np.float64(0.9244927689321858), 'AUC_Hard': np.float64(0.8414454610577566), 'Overlap': np.float64(0.24320384663858063), 'Overlap_Easy': np.float64(0.12720262151148798), 'Overlap_Hard': np.float64(0.3282081302465936)}
{'Positive': ['Douglas_fir'], 'Negative': ['Spruce_Fir', 'Cottonwood_Willow', 'Krummholz', 'Lodgepole_Pine', 'Ponderosa_Pine'], 'Easy': ['Krummholz', 'Spruce_Fir'], 'Hard': ['Lodgepole_Pine', 'Cottonwood_Willow', 'Ponderosa_Pine'], 'AUC': np.float64(0.8572072445725651), 'AUC_Easy': np.float64(0.9244927689321858), 'AUC_Hard': np.float64(0.8414454610577566), 'Overlap': np.float64(0.24320384663858063), 'Overlap_Easy': np.float64(0.12720262151148798), 'Overlap_Hard': np.float64(0.3282081302465936)}


Progress:   0%|          | 2/125000 [00:03<61:03:34,  1.76s/it]

{'Positive': ['Douglas_fir'], 'Negative': ['Spruce_Fir', 'Cottonwood_Willow', 'Krummholz', 'Lodgepole_Pine', 'Ponderosa_Pine'], 'Easy': ['Lodgepole_Pine', 'Ponderosa_Pine'], 'Hard': ['Cottonwood_Willow', 'Krummholz', 'Spruce_Fir'], 'AUC': np.float64(0.8572072445725651), 'AUC_Easy': np.float64(0.836516350807799), 'AUC_Hard': np.float64(0.9224548343013116), 'Overlap': np.float64(0.24320384663858063), 'Overlap_Easy': np.float64(0.31695239843760364), 'Overlap_Hard': np.float64(0.14345616750772394)}
{'Positive': ['Douglas_fir'], 'Negative': ['Spruce_Fir', 'Cottonwood_Willow', 'Krummholz', 'Lodgepole_Pine', 'Ponderosa_Pine'], 'Easy': ['Lodgepole_Pine', 'Ponderosa_Pine'], 'Hard': ['Cottonwood_Willow', 'Krummholz', 'Spruce_Fir'], 'AUC': np.float64(0.8572072445725651), 'AUC_Easy': np.float64(0.836516350807799), 'AUC_Hard': np.float64(0.9224548343013116), 'Overlap': np.float64(0.24320384663858063), 'Overlap_Easy': np.float64(0.31695239843760364), 'Overlap_Hard': np.float64(0.14345616750772394)}


Progress:   0%|          | 3/125000 [00:05<61:18:41,  1.77s/it]

...

Progress:   0%|          | 9/125000 [00:15<61:13:33,  1.76s/it]

{'Positive': ['Douglas_fir'], 'Negative': ['Spruce_Fir', 'Cottonwood_Willow', 'Krummholz', 'Lodgepole_Pine', 'Ponderosa_Pine'], 'Easy': ['Krummholz', 'Lodgepole_Pine', 'Ponderosa_Pine', 'Cottonwood_Willow'], 'Hard': ['Spruce_Fir'], 'AUC': np.float64(0.8572072445725651), 'AUC_Easy': np.float64(0.8354255880789597), 'AUC_Hard': np.float64(0.9315330260032411), 'Overlap': np.float64(0.24320384663858063), 'Overlap_Easy': np.float64(0.3210581411060317), 'Overlap_Hard': np.float64(0.1224850805732218)}
{'Positive': ['Douglas_fir'], 'Negative': ['Spruce_Fir', 'Cottonwood_Willow', 'Krummholz', 'Lodgepole_Pine', 'Ponderosa_Pine'], 'Easy': ['Krummholz', 'Lodgepole_Pine', 'Ponderosa_Pine', 'Cottonwood_Willow'], 'Hard': ['Spruce_Fir'], 'AUC': np.float64(0.8572072445725651), 'AUC_Easy': np.float64(0.8354255880789597), 'AUC_Hard': np.float64(0.9315330260032411), 'Overlap': np.float64(0.24320384663858063), 'Overlap_Easy': np.float64(0.3210581411060317), 'Overlap_Hard': np.float64(0.1224850805732218)}


Progress:   0%|          | 10/125000 [00:17<61:21:46,  1.77s/it]

{'Positive': ['Douglas_fir'], 'Negative': ['Spruce_Fir', 'Cottonwood_Willow', 'Krummholz', 'Lodgepole_Pine', 'Ponderosa_Pine'], 'Easy': ['Ponderosa_Pine'], 'Hard': ['Krummholz', 'Spruce_Fir', 'Lodgepole_Pine', 'Cottonwood_Willow'], 'AUC': np.float64(0.8572072445725651), 'AUC_Easy': np.float64(0.7639428299125477), 'AUC_Hard': np.float64(0.8894647022453902), 'Overlap': np.float64(0.24320384663858063), 'Overlap_Easy': np.float64(0.6178524812177459), 'Overlap_Hard': np.float64(0.14433381683705296)}
{'Positive': ['Douglas_fir'], 'Negative': ['Spruce_Fir', 'Cottonwood_Willow', 'Krummholz', 'Lodgepole_Pine', 'Ponderosa_Pine'], 'Easy': ['Ponderosa_Pine'], 'Hard': ['Krummholz', 'Spruce_Fir', 'Lodgepole_Pine', 'Cottonwood_Willow'], 'AUC': np.float64(0.8572072445725651), 'AUC_Easy': np.float64(0.7639428299125477), 'AUC_Hard': np.float64(0.8894647022453902), 'Overlap': np.float64(0.24320384663858063), 'Overlap_Easy': np.float64(0.6178524812177459), 'Overlap_Hard': np.float64(0.14433381683705296)}


KeyboardInterrupt: 

In [17]:
df

,elevation,aspect,slope,horizontal_distance_to_hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,soil_type_32,soil_type_33,soil_type_34,soil_type_35,soil_type_36,soil_type_37,soil_type_38,soil_type_39,soil_type_40,class
0,2754,146,5,150,2,1790,227,239,146,700,...,0,0,0,0,0,0,0,0,0,Lodgepole_Pine
1,3219,21,8,67,-1,2869,215,223,145,1825,...,0,0,0,0,0,0,0,0,0,Spruce_Fir
2,2965,337,16,42,7,4288,184,217,171,324,...,0,0,0,0,0,0,0,0,0,Spruce_Fir
3,2368,14,15,150,65,1006,205,208,137,812,...,0,0,0,0,0,0,0,0,0,Douglas_fir
4,2366,165,3,390,156,1165,222,240,154,582,...,0,0,0,0,0,0,0,0,0,Douglas_fir
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110388,2314,237,17,390,20,1242,187,253,200,815,...,0,0,0,0,0,0,0,0,0,Cottonwood_Willow
110389,2288,184,4,0,0,201,220,242,157,433,...,0,0,0,0,0,0,0,0,0,Cottonwood_Willow
110390,2232,141,32,85,51,1188,247,212,67,1104,...,0,0,0,0,0,0,0,0,0,Cottonwood_Willow
110391,2221,163,27,0,0,738,233,236,115,417,...,0,0,0,0,0,0,0,0,0,Cottonwood_Willow
